In [97]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict ,Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator

In [98]:
load_dotenv()

True

In [99]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [111]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="detailed feedback for the essay")
    score: float = Field(description="score out of 10", ge=0, le=10)

In [126]:
def parse_evaluation(prompt: str) -> EvaluationSchema:
    content = model.invoke(prompt).content.strip()
    if content.startswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return EvaluationSchema.model_validate_json(content)

In [102]:
essay = """ 
# The Growth of Artificial Intelligence in 2026

Artificial Intelligence (AI) has become one of the most important technologies of the 21st century. In 2026, AI is no longer limited to research laboratories or experimental applications. It has become an important part of businesses, education, healthcare, software development, finance, transportation, communication, and everyday life. The rapid development of generative AI, large language models, AI agents, robotics, and multimodal systems has significantly expanded what computers can do.

One of the biggest developments in 2026 is the growth of **generative AI**. Earlier AI systems were mainly designed to classify information, make predictions, or perform specific tasks. Modern generative AI systems can create text, images, audio, video, software code, presentations, and other forms of content. Large language models can understand natural-language instructions and assist users with writing, programming, research, learning, and problem-solving. This has changed the way people interact with computers.

Another major development is the emergence of **AI agents**. Traditional AI applications generally respond to individual requests, while AI agents can perform multiple steps to accomplish a larger objective. An agent can interpret a task, plan a sequence of actions, use tools, retrieve information, interact with software systems, and produce a result. Agentic AI is increasingly being explored for software engineering, customer support, business automation, research, data analysis, and enterprise workflows.

AI has also experienced significant growth in **healthcare**. AI systems are being used to analyze medical images, support clinical decision-making, discover potential drugs, analyze medical data, and assist healthcare professionals. AI can help process large amounts of information quickly, although human expertise, clinical validation, privacy, and safety remain essential when AI is used in healthcare.

In **education**, AI has created new possibilities for personalized learning. Students can use AI tutors to receive explanations, practice questions, feedback, summaries, and programming assistance. Teachers can use AI to prepare educational materials and support administrative tasks. However, educational institutions also need to address issues such as academic integrity, overdependence on AI, inaccurate information, and responsible use.

The **software industry** has also been strongly affected by AI. AI-powered programming tools can generate code, explain existing programs, identify potential errors, write tests, and assist developers with documentation. This does not necessarily eliminate the need for software engineers. Instead, it is changing the skills required from developers. Understanding programming fundamentals, system design, security, testing, databases, and AI-assisted development is becoming increasingly important.

AI is also transforming **business and industry**. Organizations are using AI for customer service, marketing, fraud detection, forecasting, supply-chain optimization, document processing, and decision support. Companies can combine AI with existing enterprise systems to automate repetitive tasks and improve access to information. Retrieval-Augmented Generation (RAG), for example, allows AI systems to retrieve relevant information from organizational documents and use that information when generating responses.

The growth of AI has also increased demand for **AI infrastructure**. Modern AI applications require powerful computing resources, specialized processors, large datasets, efficient data pipelines, and reliable cloud infrastructure. This has increased the importance of technologies such as GPUs, cloud computing, distributed systems, model optimization, and efficient inference.

Despite its rapid growth, AI also presents significant challenges. AI systems can produce incorrect information, reflect biases present in training data, expose sensitive information, or be misused. Organizations therefore need strong approaches to privacy, cybersecurity, transparency, evaluation, and human oversight. Governments and technology companies are also developing policies and frameworks intended to encourage responsible development and deployment of AI.

Another important concern is the effect of AI on employment. AI can automate certain repetitive tasks and change the responsibilities associated with many jobs. At the same time, it is creating new opportunities in areas such as AI engineering, machine learning, data science, AI security, AI product development, and AI governance. The ability to work effectively with AI tools is therefore becoming an increasingly valuable professional skill.

The growth of AI in 2026 can also be seen as a transition from **AI as a tool to AI as a collaborator**. People increasingly interact with AI through natural language rather than traditional software interfaces. Instead of manually performing every step, users can describe an objective and allow AI systems to assist with planning and execution. This represents an important change in how software may be designed and used.

However, the future development of AI should not be measured only by how powerful models become. Reliability, safety, affordability, energy efficiency, privacy, transparency, and real-world usefulness are equally important. A highly capable AI system that produces unreliable results can create significant risks. Therefore, responsible development and effective human oversight must grow alongside AI capabilities.

In conclusion, **2026 represents an important stage in the evolution of Artificial Intelligence**. Generative AI, multimodal models, AI agents, RAG systems, robotics, and AI-powered software development are expanding the role of intelligent systems across society. AI is creating new opportunities for businesses, students, researchers, and professionals while also introducing challenges related to employment, privacy, security, reliability, and ethics. The future of AI will depend not only on technological innovation but also on how responsibly individuals, organizations, and governments choose to develop and use it. AI is therefore not simply a technology of the future—it is becoming an important part of the present.

"""

In [ ]:
prompt = f'evaluate the language quality of the following essay and provide valid JSON with the keys feedback and score.\n{essay}'
EvaluationSchema.model_validate_json(model.invoke(prompt).content)

EvaluationSchema(feedback='The essay is clear, well‑structured, and covers a wide range of AI developments in 2026. It uses a consistent formal tone and appropriate technical vocabulary, which makes it accessible to a general audience while still conveying depth. The paragraphing is logical, and the use of bold headings helps guide the reader. Minor issues include occasional repetition (e.g., “AI is becoming an important part of the present” echoes earlier statements), some slightly wordy phrases (“AI systems are being used to analyze medical images, support clinical decision‑making, discover potential drugs, analyze medical data, and assist healthcare professionals”), and a few instances of awkward phrasing (“AI can help process large amounts of information quickly, although human expertise, clinical validation, privacy, and safety remain essential when AI is used in healthcare”). These do not significantly hinder comprehension but could be tightened for concision. Overall, the langua

In [113]:
class UPSEState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[float], operator.add]
    avg_score: float

In [121]:
def evaluate_language(state:UPSEState):
    prompt = f'evaluate the language quality of the following essay and provide feedback and a score out of 10. Return only valid JSON with the keys feedback and score.\n{state["essay"]}'
    output = parse_evaluation(prompt)

    return {'language_feedback':output.feedback,'individual_score':[output.score]}

In [122]:
def evaluate_analysis(state:UPSEState):
    prompt = f'evaluate the depth of analysis of the following essay and provide feedback and a score out of 10. Return only valid JSON with the keys feedback and score.\n{state["essay"]}'
    output = parse_evaluation(prompt)

    return {'analysis_feedback':output.feedback,'individual_score':[output.score]}

In [123]:
def evaluate_thought(state:UPSEState):
    prompt = f'evaluate the clarity of thought of the following essay and provide feedback and a score out of 10. Return only valid JSON with the keys feedback and score.\n{state["essay"]}'
    output = parse_evaluation(prompt)

    return {'clarity_feedback':output.feedback,'individual_score':[output.score]}

In [117]:
def final_eval(state:UPSEState):
    # summary feedback
    prompt = f"based on the following feedback, create a summarized feedback:\nlanguage feedback - {state['language_feedback']}\ndepth of analysis feedback - {state['analysis_feedback']}\nclarity of thought feedback - {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content

    # average score
    avg_score = sum(state['individual_score'])/len(state['individual_score'])

    return {'overall_feedback':overall_feedback,'avg_score':avg_score}

In [127]:
graph = StateGraph(UPSEState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_eval',final_eval)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_eval')
graph.add_edge('evaluate_analysis','final_eval')
graph.add_edge('evaluate_thought','final_eval')

graph.add_edge('final_eval',END)

workfolw = graph.compile()

In [129]:
initial_state = {
    'essay':essay
}
workfolw.invoke(initial_state)

{'essay': ' \n# The Growth of Artificial Intelligence in 2026\n\nArtificial Intelligence (AI) has become one of the most important technologies of the 21st century. In 2026, AI is no longer limited to research laboratories or experimental applications. It has become an important part of businesses, education, healthcare, software development, finance, transportation, communication, and everyday life. The rapid development of generative AI, large language models, AI agents, robotics, and multimodal systems has significantly expanded what computers can do.\n\nOne of the biggest developments in 2026 is the growth of **generative AI**. Earlier AI systems were mainly designed to classify information, make predictions, or perform specific tasks. Modern generative AI systems can create text, images, audio, video, software code, presentations, and other forms of content. Large language models can understand natural-language instructions and assist users with writing, programming, research, lea